<a href="https://colab.research.google.com/github/CIS5190/cis-4190-5190-fall-26/blob/main/hw2/hw2_skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS 4190/5190 Fall 2026 - Homework 2**

# **Tips for Coding Homeworks**
Dear class, here are some tips to follow in coding HWs in this class, to minimize the chances of hitting hard-to-resolve bugs:

1.   **Do not set/alter the random seeds we provide.** (Seeds fixed in setup or "don't change this line" cells exist so that everyone's numbers, and the hidden autograder tests, match up.)
2.   Most bugs come down to minor errors in your code. The hardest and most important part of coding is debugging; debugging ML code is a key skillset you should acquire in this class.
3.   Good python coding avoids loops if at all possible: in particular, **for any vector and matrix operations, use `numpy` functions instead of loops**. This will often speed up your code by orders of magnitude. You may also want to take a look at the [python primer](https://www.engineering.upenn.edu/~cis5190/fall2026/docs/basics_python_numpy.ipynb).
4.   Pay **extra attention to "copying" vs "moving" objects** i.e. take care during assignment operations. Otherwise, it may lead to incorrect results or Out Of Memory issues and Colab notebook restarts.
5.   **Each graded function is run on its own.** The autograder takes the function you wrote and runs it by itself, with only `np`, `pd`, `math` and `random` available. It never sees your other cells. So if a graded function needs another import or a helper function, put it *inside* the function: an import at the top of the notebook, or a helper defined in another cell, gives a `NameError` and no marks for that question.
6.   For this homework especially, pay **special attention to array shapes**. Several questions grade a prediction array, and an array of shape `(N, 1)` is not the same thing as one of shape `(N,)`. Each question states the shape it expects.

Have fun!

> **Important: Save a Copy Before Starting**
> Please click *"Copy to Drive"* in the top menu bar or go to *File → Save a Copy in Drive*.
> This shared copy is read-only, so any edits you make to it will not be saved. Work in your
> own copy.

---

> **Points**
> This assignment is worth **50 points**: 43 points that every student completes, plus 7
> points marked **[5190 Extra]**, which are section 5.2 (confidence intervals, 3 pts)
> and section 5.3 (the performance table, 4 pts).
>
> - **CIS 4190** is graded out of **43**. The [5190 Extra] parts are optional for you.
> - **CIS 5190** is graded out of **50**. The [5190 Extra] parts are required for you.
>
> CIS 4190 students are welcome to attempt the [5190 Extra] parts for credit, but your total
> for this assignment is still capped at 43 points.

---

> **How this notebook is graded**
> 39 of the 50 points are graded by the autograder in this notebook, from the
> PennGrader cells. The other 11 are graded by hand from the plots and written answers
> you leave in the notebook — sections 1.3, 4.1.3, 5.3 and 6. **Download the notebook with
> its outputs intact**, or there will be nothing there to mark.
>
> Each grading cell is marked on its own, from what that cell sends. Tip 5 above explains what
> that means for the functions you write.
>
> **Your most recent submission for each grading cell is the one that counts** — re-running a
> grading cell with a worse answer will lower your recorded score. You may resubmit each
> grading cell up to 100 times per day.

## Setup Cell 1: Imports and Configuration

This cell imports essential libraries you will need for this assignment. It also disables Pandas' chained assignment warnings to reduce clutter in your output.

*Note:* Only import libraries when you need them to keep your notebook clean. Additional imports will be added as the assignment progresses.

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress pandas chained assignment warnings for cleaner output
pd.options.mode.chained_assignment = None

np.random.seed(42)  # don't change this line!!
random.seed(42)     # don't change this line!!

## Setup Cell 2: Environment Detection

This cell installs the `dill` package (used to retrieve source code) and detects whether the notebook is running in Google Colab or the autograder environment.

- `NOTEBOOK` will be `True` if running in Colab and `False` in the autograder.
- This check helps your code run correctly depending on the environment.
- Do **not** modify this cell.

In [ ]:
# For autograder only, do not modify this cell.
# Determines if code is running in Google Colab or autograder environment

!pip install dill
from dill.source import getsource

NOTEBOOK = (os.getenv('IS_AUTOGRADER') is None)

if NOTEBOOK:
    print("[INFO, OK] Google Colab.")
else:
    print("[INFO, OK] Autograder.")

## PennGrader Setup

As in HW0 and HW1, you'll use PennGrader to get instant feedback on your answers.

Because Part 3 asks you to submit entire **classes** (not just single functions), we need the
helper cell below (`grader_serialize`) that knows how to pull a full class definition out of
your notebook so PennGrader can grade it.

In [ ]:
%%capture
!pip3 install penngrader-client==0.5.13

In [ ]:
from penngrader.grader import *

### Autograder helpers

This cell holds two helpers. One lets PennGrader read the source of a **class** you define; the
other lets the local tests check that a function doesn't use a library you've been asked not
to use. Do **not** modify this cell.

In [ ]:
# For autograder only, do not modify this cell.
# Lets PennGrader retrieve the full source of a class defined in an earlier cell.

import inspect, sys
from IPython.core.magics.code import extract_symbols

def new_getfile(object, _old_getfile=inspect.getfile):
    if not inspect.isclass(object):
        return _old_getfile(object)

    # Lookup by parent module (as in current inspect)
    if hasattr(object, '__module__'):
        object_ = sys.modules.get(object.__module__)
        if hasattr(object_, '__file__'):
            return object_.__file__

    # If parent module is __main__, lookup by methods
    for name, member in inspect.getmembers(object):
        if inspect.isfunction(member) and object.__qualname__ + '.' + member.__name__ == member.__qualname__:
            return inspect.getfile(member)
    else:
        raise TypeError('Source for {!r} not found'.format(object))
inspect.getfile = new_getfile

def grader_serialize(obj):
    cell_code = "".join(inspect.linecache.getlines(new_getfile(obj)))
    class_code = extract_symbols(cell_code, obj.__name__)[0][0]
    return class_code


import ast

def _code_uses(src, name):
    # True if the code in src uses `name`: imports it, refers to it, or reaches it as
    # an attribute. Comments and strings don't count. The autograder runs the same check.
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return False
    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and node.id == name:
            return True
        if isinstance(node, ast.Attribute) and node.attr == name:
            return True
        if isinstance(node, ast.Import) and any(name in a.name.split(".") for a in node.names):
            return True
        if isinstance(node, ast.ImportFrom) and (
                name in (node.module or "").split(".") or any(a.name == name for a in node.names)):
            return True
    return False

### Autograder Setup

Please enter your **8-digit PennID** in the cell below:

In [ ]:
# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY.
# IF NOT, THE AUTOGRADER WON'T KNOW WHO TO ASSIGN POINTS TO IN OUR BACKEND.

STUDENT_ID = 12345678          # YOUR PENN-ID GOES HERE AS AN INTEGER

## Initialize the Autograder

Run the following cell to initialize the autograder. This autograder will let you submit your code directly from this notebook and immediately receive a score.

**NOTE:** We store your submissions and check them against other students' submissions. So, not that you would, but please do **not** cheat.

In [ ]:
%%writefile notebook-config.yaml

grader_api_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/TestSandbox"
grader_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"
token_generator_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/GetTokens"
token_generator_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"


In [ ]:
COURSE_NAME = 'cis5190_fall2026'
HOMEWORK_NUMBER = 2   # -> homework ID cis5190_fall2026_HW2

grader = PennGrader('notebook-config.yaml', HOMEWORK_NUMBER, STUDENT_ID, STUDENT_ID, COURSE_NAME)

# **1. K-Nearest Neighbours [6 pts]**

While doing classification, KNN searches the memorized training instances for the $k$ instances that most closely resemble the new instance and assigns to it the most common class. An alternate way of understanding KNN is by looking at the learned decision boundaries. In this problem, you will classify points in the X-Y plane using [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html). The training dataset is the [Iris dataset](https://archive.ics.uci.edu/dataset/53/iris), and each point in the 2D space is classified into one of three species from its x-coordinate (sepal length) and y-coordinate (sepal width).

## **1.1. Load the Iris Dataset [ungraded]**

Complete `load_iris_dataset` to

- populate `X_train` with the Iris features. Use only sepal length and width, i.e. the first two columns, giving shape `(m, 2)`
- populate `y_train` with the labels (species), shape `(m,)`
- return `X_train, y_train`

The dataset ships inside `scikit-learn`, so there is nothing to download.

In [ ]:
from sklearn import datasets

def load_iris_dataset():
    """Load the Iris features and labels used throughout Part 1.

    Returns:
        X_train: np.ndarray of shape (m, 2) -- sepal length and sepal width only
        y_train: np.ndarray of shape (m,)   -- the species label, 0, 1 or 2
    """
    # Import training data
    iris = datasets.load_iris()

    # STUDENT TODO START:
    # Examine `iris` and build the two arrays described above, then return them.
    raise NotImplementedError("load_iris_dataset is not implemented yet.")
    # STUDENT TODO END


# Load the iris dataset first
X_train, y_train = load_iris_dataset()

## **1.2. Standardize the Features [3 pts, Autograded]**

Complete `standardise_features` to standardize the features by subtracting the mean and
scaling to unit variance, i.e.

$$z = \frac{x - u}{s}$$

where $u$ is the mean of the training data and $s$ is its standard deviation. This is the
standardization from lecture (L05).

**Notes**:
* Centering and scaling happen independently on each feature (column).
* Use the **population** standard deviation — `np.std`'s default, `ddof=0`.
* Implement this yourself: **do not use sklearn's `StandardScaler`**. A solution that uses it
  gets no credit. (Mentioning it in a comment is fine.)
* You are encouraged to use `numpy` vectorization/broadcasting rather than loops.

Marks: **3** — 1 for returning an array of the right shape, 2 for the values being right.

In [ ]:
def standardise_features(X_train):
    """Standardize each column of X_train to zero mean and unit variance.

    Args:
        X_train: np.ndarray of shape (m, n)

    Returns:
        np.ndarray of shape (m, n), the standardized features
    """
    # STUDENT TODO START:
    raise NotImplementedError("standardise_features is not implemented yet.")
    # STUDENT TODO END


# --- Local test. It checks mean 0 and standard deviation 1 in each column, and the exact
# values on this small example. The hidden test compares values on a different array.
_check = standardise_features(np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0], [4.0, 40.0]]))
assert _check.shape == (4, 2), f"Expected shape (4, 2), got {_check.shape}"
assert np.allclose(np.mean(_check, axis=0), 0.0), "Each column should have mean 0 after standardizing."
assert np.allclose(np.std(_check, axis=0), 1.0), "Each column should have standard deviation 1 (ddof=0)."
assert np.allclose(_check[:, 0], [-1.34164079, -0.4472136, 0.4472136, 1.34164079]),     "Mean 0 and std 1 aren't enough on their own: each value should be (x - mean) / std."
assert not _code_uses(grader_serialize(standardise_features), "StandardScaler"),     "1.2 asks you to write this yourself, so a solution that uses StandardScaler gets no credit."
print("1.2 local test passed.")

X_train = standardise_features(X_train)

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_knn_standardise', answer=grader_serialize(standardise_features))

## **1.3. Plot the KNN Decision Boundary [3 pts, Manually Graded]**

Complete `plot_KNN_boundary` to:
* train a KNN classifier with `k` neighbours on the provided `X_train` and `y_train`
* predict on `X_test` and store the result in `y_test`

Then run the cell below it, which calls your function for several values of `k`, and answer
the question underneath.

Marks: **1** for the classifier and predictions, **1** for the four plots rendering, **1** for your
written answer.

This section is graded by hand from the plots and the comment in this notebook, so leave both
in place when you download it.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier

def plot_KNN_boundary(k, X_train, y_train):
    """Fit a k-NN classifier and plot its decision boundary over the training points.

    Args:
        k: number of neighbours to use for k-neighbours queries
        X_train: training features, shape (m, 2)
        y_train: training labels, shape (m,)

    Returns:
        None -- the function draws a plot.
    """
    # Mesh the 2-D space into a grid to generate X_test
    h = 0.02
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    X_test = np.c_[xx.ravel(), yy.ravel()]
    y_test = np.zeros(xx.shape)

    # STUDENT TODO START:
    # 1. train a KNN classifier with k neighbours
    # 2. save its predictions on X_test in y_test
    raise NotImplementedError("plot_KNN_boundary is not implemented yet.")
    # STUDENT TODO END

    # Put the result into a color plot
    y_test = y_test.reshape(xx.shape)
    cmap_light = ListedColormap(['orange', 'cyan', 'cornflowerblue'])
    cmap_bold = ['darkorange', 'c', 'darkblue']
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, y_test, cmap=cmap_light)

    # Also plot the training points
    iris_target_names = ['setosa', 'versicolor', 'virginica']
    sns.scatterplot(x=X_train[:, 0], y=X_train[:, 1],
                    hue=[iris_target_names[y] for y in y_train],
                    palette=cmap_bold, alpha=1.0, edgecolor="black")
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title("3-Class classification (k = %i)" % (k))
    plt.xlabel("standardised sepal length")
    plt.ylabel("standardised sepal width")
    plt.show()

### Explore the effect of changing $k$ on the learned decision boundaries.

In [ ]:
# Plot KNN decision boundaries with different k values
def visualize_KNN(X_train, y_train):
    k_list = [1, 4, 11, 15]

    # STUDENT TODO START:
    # Call plot_KNN_boundary for each value of k in k_list.
    raise NotImplementedError("visualize_KNN is not implemented yet.")
    # STUDENT TODO END


visualize_KNN(X_train, y_train)

**Your comment (1.3):** Looking at the four plots above, does the model **underfit** or
**overfit** as $k$ increases, and why? Two or three sentences.

*replace this italic line with your answer*

# **2. The Diabetes Dataset [3 pts]**

Parts 3 and 5 both predict the presence of diabetes from survey and examination data, so we
load and clean that dataset once, here.

The data is from the [2013–2014 National Health and Nutrition Examination Survey
(NHANES)](https://wwwn.cdc.gov/nchs/nhanes/Default.aspx) — the same survey used in the
decision-tree lecture. We have reduced it to 20 features; the original had over 1,800.
`cis5190_hw2_diabetes_train.csv` carries the labels, `cis5190_hw2_diabetes_X_test.csv` does not:
**the test labels are held by the autograder**, which is how your predictions get scored.

## **2.1. Load the Datasets [ungraded]**

Read `cis5190_hw2_diabetes_train.csv` into `train_df` and `cis5190_hw2_diabetes_X_test.csv`
into `test_df`.

In [ ]:
import os

if not os.path.exists("cis5190_hw2_diabetes_train.csv"):
    !wget https://raw.githubusercontent.com/CIS5190/cis-4190-5190-fall-26/main/hw2/cis5190_hw2_diabetes_train.csv
if not os.path.exists("cis5190_hw2_diabetes_X_test.csv"):
    !wget https://raw.githubusercontent.com/CIS5190/cis-4190-5190-fall-26/main/hw2/cis5190_hw2_diabetes_X_test.csv

In [ ]:
import pandas as pd

def load_diabetes_datasets():
    """Read the two CSVs sitting next to this notebook.

    Returns:
        train_df: the training set, including the DIABETIC column
        test_df:  the test features, with no DIABETIC column
    """
    # STUDENT TODO START:
    raise NotImplementedError("load_diabetes_datasets is not implemented yet.")
    # STUDENT TODO END


train_df, test_df = load_diabetes_datasets()
print("raw train:", train_df.shape, " raw test:", test_df.shape)

## **2.2. Preprocess the Datasets [3 pts, Autograded]**

The data is not ready to use: it has columns we do not want and a great many missing values.
Complete `preprocess_datasets` to do both jobs, in this order.

**Feature selection.** Keep exactly these 13 features, in this order, and drop every other column:

`RIDAGEYR`, `BMXWAIST`, `BMXHT`, `LBXTC`, `BMXLEG`, `BMXWT`, `BMXBMI`, `RIDRETH1`, `BPQ020`,
`ALQ120Q`, `DMDEDUC2`, `RIAGENDR`, `INDFMPIR`

`train_df` keeps a 14th column, `DIABETIC`, as the target. It must be the **last** column, since
later sections address it as `train_df.iloc[:, -1]`. `test_df` has no target column.

**Missing values.**
* In the **training** set, drop every row that still has a missing value.
* In the **test** set, do *not* drop rows — the autograder expects a prediction for all 2035 of
  them. Impute each missing value with the **mean of that column in the already-cleaned
  training set**.

> **Select the features first, then drop rows.** Several of the 20 original columns are missing
> for almost every respondent, so dropping rows on the full table leaves you with an empty
> training set.

Marks: **3** — 1 for the columns, 1 for the training set and 1 for the test set. The
autograder checks the values, not just the shapes. For example, filling the test gaps with 0
instead of the training mean loses the test-set point.

In [ ]:
# The 13 features every later section works from.
REQUIRED_FEATURES = ['RIDAGEYR', 'BMXWAIST', 'BMXHT', 'LBXTC', 'BMXLEG', 'BMXWT', 'BMXBMI', 'RIDRETH1', 'BPQ020', 'ALQ120Q', 'DMDEDUC2', 'RIAGENDR', 'INDFMPIR']

In [ ]:
def preprocess_datasets(train_df, test_df):
    """Select the 13 required features and handle missing values.

    Args:
        train_df: the raw training set
        test_df:  the raw test features

    Returns:
        train_df: shape (n_train, 14) -- the 13 features plus DIABETIC last, no missing values
        test_df:  shape (2035, 13)    -- the 13 features, no missing values, no rows dropped
    """
    # STUDENT TODO START:
    raise NotImplementedError("preprocess_datasets is not implemented yet.")
    # STUDENT TODO END


_test_gaps = test_df[REQUIRED_FEATURES].isna()   # where the raw test set has gaps

train_df_processed, test_df_processed = preprocess_datasets(train_df, test_df)

# --- Local test. The hidden test checks the same things, and also compares your
# column means with the right ones.
assert list(test_df_processed.columns) == REQUIRED_FEATURES,     "test_df should hold exactly the 13 required features, in the order listed above."
assert list(train_df_processed.columns) == REQUIRED_FEATURES + ["DIABETIC"],     "train_df should hold the 13 required features followed by DIABETIC."
assert train_df_processed.isna().sum().sum() == 0, "train_df still has missing values."
assert test_df_processed.isna().sum().sum() == 0, "test_df still has missing values."
assert test_df_processed.shape == (2035, 13),     f"test_df should be (2035, 13) -- do not drop test rows. Got {test_df_processed.shape}."
assert len(train_df_processed) == 3000,     f"The cleaned training set should have 3000 rows, but has {len(train_df_processed)}."
_means = train_df_processed[REQUIRED_FEATURES].mean()
for _c in REQUIRED_FEATURES:
    assert np.allclose(test_df_processed[_c].values[_test_gaps[_c].values], _means[_c]),         f"The gaps in the test set's {_c} column should be filled with {_c}'s mean in the cleaned training set."
print("2.2 local test passed. train:", train_df_processed.shape, " test:", test_df_processed.shape)

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_preprocess_datasets',
             answer=(train_df_processed.to_csv(index=False), test_df_processed.to_csv(index=False)))

# **3. Neural Networks [13 pts]**

You will build a small feed-forward network from its parts and train it on the diabetes data
you cleaned in Part 2.

## **3.0. Neural Network Playground [ungraded]**

Before you write any code, it is worth a few minutes with TensorFlow's
[Neural Network Playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2,2,2,2,2&seed=0.32529&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false).

Change the number of hidden layers, the neurons per layer, the activation function, the
learning rate and the regularization rate, and watch what happens to the decision boundary.
Nothing here is graded or submitted — it is the fastest way to build intuition for the
hyperparameters you are about to set by hand.

## **3.1. Code a Simple Neural Network Model [8 pts, Autograded]**

In lecture (L09) you saw a layer written as $a^{(l)} = g_l\!\left(W^{(l)} a^{(l-1)} + b^{(l)}\right)$, and in
L10 you worked backpropagation through a computation graph by hand. Here you will implement one
such layer — both directions — and then stack three of them.

PyTorch's **autograd** normally computes the backward pass for you. We are going to switch that
off for the linear layer and write it ourselves, so that the chain rule you did on paper and
the code that runs on a GPU are visibly the same thing. You define a new autograd operator by
subclassing `torch.autograd.Function` and implementing two static methods:

* `forward(ctx, ...)` computes output tensors from input tensors.
* `backward(ctx, grad_output)` receives $\partial L/\partial \text{output}$ and returns
  $\partial L / \partial \text{input}$ for each input `forward` took.

Reference: the PyTorch [examples walkthrough](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html)
and [extending autograd](https://pytorch.org/docs/stable/notes/extending.html#extending-torch-autograd).

Marks: **5** for `CustomLinearFunction` (2 forward, 3 backward — one per gradient) and
**3** for `CustomLinear` and `CustomNeuralNetwork` (1 for the layer's parameters, 1 for its
forward pass going through `CustomLinearFunction`, 1 for the network's forward pass).

You may use `torch` throughout Part 3. You may **not** use `torch.nn.Linear` — writing it is
the exercise.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)  # don't change this line!!

In [ ]:
# Firstly, we will implement a custom autograd operator.
# Note that we want to calculate output in the forward pass as:
#     output = input x transpose(weight) + bias

class CustomLinearFunction(torch.autograd.Function):
    """A custom autograd Function for a linear transformation.

    Computes the linear transformation in the forward pass and computes the
    gradients by hand in the backward pass.
    """

    @staticmethod
    def forward(ctx, input, weight, bias):
        """Performs the forward pass of the linear layer.

        Args:
            ctx: a context object used to stash tensors for the backward pass
            input: torch.Tensor of shape (N, in_features)
            weight: torch.Tensor of shape (out_features, in_features)
            bias: torch.Tensor of shape (out_features,)

        Returns:
            output: torch.Tensor of shape (N, out_features)
        """
        # Save tensors input, weight and bias for backward pass
        ctx.save_for_backward(input, weight, bias)

        # STUDENT TODO START:
        # Compute output = input @ weight.T + bias and assign it to `output`.
        raise NotImplementedError("CustomLinearFunction.forward is not implemented yet.")
        # STUDENT TODO END

        # Return the output
        return output

    @staticmethod
    def backward(ctx, grad_output):
        """Performs the backward pass of the linear layer.

        Args:
            ctx: the context object holding the tensors saved by forward()
            grad_output: torch.Tensor of shape (N, out_features), the gradient of
                the loss with respect to this layer's output

        Returns:
            grad_input:  torch.Tensor of shape (N, in_features), or None
            grad_weight: torch.Tensor of shape (out_features, in_features), or None
            grad_bias:   torch.Tensor of shape (out_features,), or None

        Each returned gradient must have the same shape as the input it corresponds
        to, and must be None when that input does not require a gradient.
        """
        # Retrieve saved tensors
        input, weight, bias = ctx.saved_tensors

        # Initialize gradients to None
        grad_input = grad_weight = grad_bias = None

        # STUDENT TODO START:
        # ctx.needs_input_grad is a tuple of booleans, one per input to forward().
        # Guard each gradient with its entry, then compute it.
        raise NotImplementedError("CustomLinearFunction.backward is not implemented yet.")
        # STUDENT TODO END

        # Return gradients in the same order as inputs to forward()
        return grad_input, grad_weight, grad_bias

In [ ]:
# Next we will have to build a custom linear layer

class CustomLinear(nn.Module):
    """A linear layer built on CustomLinearFunction.

    Attributes:
        weight: nn.Parameter of shape (out_features, in_features)
        bias: nn.Parameter of shape (out_features,)
    """

    def __init__(self, in_features, out_features):
        super(CustomLinear, self).__init__()

        # STUDENT TODO START:
        # Create self.weight as an nn.Parameter of shape (out_features, in_features).
        # Use torch.empty -- reset_parameters() below fills it in.
        raise NotImplementedError("CustomLinear.__init__ is not implemented yet.")
        # STUDENT TODO END

        self.bias = nn.Parameter(torch.empty(out_features))
        # Initialize parameters
        self.reset_parameters()

    def reset_parameters(self):
        """Initialize the layer's parameters.

        Draw the weights from a normal distribution with **mean 0.0 and standard
        deviation 0.02**, and set the biases to exactly zero. Use `nn.init.normal_`
        and `nn.init.zeros_`.

        The standard deviation is not a free choice: section 3.2's accuracy and F1
        thresholds were set against 0.02, so a much larger value will train to a
        different place.
        """
        # STUDENT TODO START:
        raise NotImplementedError("CustomLinear.reset_parameters is not implemented yet.")
        # STUDENT TODO END

    def forward(self, input):
        """Apply the custom linear transformation to the input.

        Args:
            input: torch.Tensor of shape (N, in_features)

        Returns:
            torch.Tensor of shape (N, out_features)
        """
        # Use the custom autograd function -- call it with .apply(...), not directly.
        # The autograder checks that the output goes through CustomLinearFunction.
        # STUDENT TODO START:
        raise NotImplementedError("CustomLinear.forward is not implemented yet.")
        # STUDENT TODO END

In [ ]:
# Now, we will build our NN

class CustomNeuralNetwork(nn.Module):
    """A network with two hidden layers built from CustomLinear layers.

    Args:
        input_size: number of input features
        hidden_size1: neurons in the first hidden layer
        hidden_size2: neurons in the second hidden layer
        output_size: number of output neurons
    """

    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(CustomNeuralNetwork, self).__init__()
        # Define layers using the CustomLinear module
        self.layer1 = CustomLinear(input_size, hidden_size1)
        self.layer2 = CustomLinear(hidden_size1, hidden_size2)
        self.output_layer = CustomLinear(hidden_size2, output_size)

    def forward(self, x):
        """Run the forward pass.

        Apply layer1 then a ReLU, layer2 then a ReLU, and the output layer then a
        **sigmoid** -- section 3.2 trains this with `nn.BCELoss`, which expects
        probabilities in [0, 1], not logits.

        Args:
            x: torch.Tensor of shape (N, input_size)

        Returns:
            torch.Tensor of shape (N, output_size)
        """
        # STUDENT TODO START:
        raise NotImplementedError("CustomNeuralNetwork.forward is not implemented yet.")
        # STUDENT TODO END

### Check your layer, then submit it

The cell below runs your three classes on small fixed examples and records what they return.
The two grading cells after it send those records to the autograder. Don't change this cell.

The checks at the bottom catch the usual mistakes, but they aren't the whole hidden test, which
also compares your numbers with the right ones. If one of your methods raises an error, the
cell records it and carries on, so one broken method doesn't stop the others from being
graded.

In [ ]:
# For autograder only, do not modify this cell.
# Runs your classes on small fixed examples and records what they return. If one of your
# methods raises, its error is recorded in its place and the rest still run.

class _Ctx:
    """Stands in for autograd's ctx, so forward() and backward() can be called directly
    and each gradient is recorded exactly as your backward() returns it."""
    needs_input_grad = (True, True, True)

    def save_for_backward(self, *tensors):
        self.saved_tensors = tensors


def _record(get):
    try:
        return get()
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"


def _as_list(t, what):
    if t is None:
        raise ValueError(f"{what} is None")
    return t.detach().numpy().tolist()


# Fixed inputs. Every value is a multiple of 0.25, so it is exact on any machine.
_X = [[0.5, -1.5, 2.0], [-0.25, 1.0, -0.75], [1.25, 0.0, -2.5], [-1.0, 0.5, 1.5]]
_W = [[0.75, -0.5, 0.25], [-1.25, 1.5, -0.75]]
_B = [0.5, -0.25]


def _nn_probe():
    x, w, b = (torch.tensor(v, dtype=torch.float64) for v in (_X, _W, _B))
    upstream = torch.arange(1.0, 9.0, dtype=torch.float64).reshape(4, 2)
    ctx = _Ctx()
    probe = {"forward": _record(
        lambda: _as_list(CustomLinearFunction.forward(ctx, x, w, b), "forward's output"))}
    grads = _record(lambda: CustomLinearFunction.backward(ctx, upstream))
    for i, key in enumerate(("grad_input", "grad_weight", "grad_bias")):
        probe[key] = grads if isinstance(grads, str) else _record(lambda: _as_list(grads[i], key))
    return probe


def _network_probe():
    torch.manual_seed(0)
    layer = _record(lambda: CustomLinear(100, 50))
    probe = {}
    for key, get in (
        ("weight_shape", lambda: list(layer.weight.shape)),
        ("bias_is_zero", lambda: bool(torch.all(layer.bias == 0).item())),
        ("weight_std", lambda: float(torch.std(layer.weight).item())),
        ("weight_mean", lambda: float(torch.mean(layer.weight).item())),
    ):
        probe[key] = layer if isinstance(layer, str) else _record(get)

    def set_params(lyr, out_f, in_f):
        lyr.weight = nn.Parameter(torch.linspace(-0.5, 0.5, out_f * in_f).reshape(out_f, in_f))
        lyr.bias = nn.Parameter(torch.linspace(0.1, 0.3, out_f))
        return lyr

    def forward_node():
        small = set_params(CustomLinear(3, 2), 2, 3)
        return type(small(torch.ones(1, 3)).grad_fn).__name__

    def run_network():
        net = CustomNeuralNetwork(3, 4, 2, 1)
        for lyr, (out_f, in_f) in ((net.layer1, (4, 3)), (net.layer2, (2, 4)),
                                   (net.output_layer, (1, 2))):
            set_params(lyr, out_f, in_f)
        with torch.no_grad():
            return _as_list(net(torch.tensor([[0.2, -0.4, 0.6], [1.0, 0.0, -1.0]])),
                            "the network's output")

    probe["forward_node"] = _record(forward_node)
    probe["network_forward"] = _record(run_network)
    return probe


NN_PROBE = _nn_probe()
NETWORK_PROBE = _network_probe()

# --- Local checks. They catch the usual mistakes; the hidden tests also compare your
# numbers with the right ones.
_errors = {k: v for k, v in {**NN_PROBE, **NETWORK_PROBE}.items()
           if isinstance(v, str) and v.startswith("error")}
_listing = "\n".join(f"  {k}: {v[len('error: '):]}" for k, v in _errors.items())
if _errors and all("NotImplementedError" in v for v in _errors.values()):
    raise NotImplementedError("Some of your 3.1 methods aren't written yet:\n" + _listing)
assert not _errors, "Some of your methods raised an error:\n" + _listing

assert np.allclose(NN_PROBE["forward"], np.array(_X) @ np.array(_W).T + np.array(_B)), \
    "forward() should compute input @ weight.T + bias."
for _key, _shape in (("grad_input", (4, 3)), ("grad_weight", (2, 3)), ("grad_bias", (2,))):
    assert np.shape(NN_PROBE[_key]) == _shape, \
        f"{_key} should have shape {_shape}, the same as the input it belongs to. Got {np.shape(NN_PROBE[_key])}."
try:
    torch.autograd.gradcheck(CustomLinearFunction.apply, tuple(
        torch.tensor(v, dtype=torch.float64, requires_grad=True) for v in (_X, _W, _B)))
except Exception as _e:
    raise AssertionError(
        "backward() doesn't agree with PyTorch's numerical check of the gradients. In the "
        "message below, input 0 is `input`, 1 is `weight` and 2 is `bias`.\n\n" + str(_e)) from None

assert NETWORK_PROBE["weight_shape"] == [50, 100], \
    f'CustomLinear(100, 50).weight should be (out_features, in_features) = (50, 100), got {tuple(NETWORK_PROBE["weight_shape"])}.'
assert NETWORK_PROBE["bias_is_zero"], "reset_parameters should set the bias to exactly zero."
assert 0.019 <= NETWORK_PROBE["weight_std"] <= 0.021 and abs(NETWORK_PROBE["weight_mean"]) <= 0.003, \
    (f'The weights should come from a normal distribution with mean 0 and std 0.02; yours have '
     f'mean {NETWORK_PROBE["weight_mean"]:.4f} and std {NETWORK_PROBE["weight_std"]:.4f}.')
assert NETWORK_PROBE["forward_node"].startswith("CustomLinearFunction"), \
    "CustomLinear.forward should call CustomLinearFunction.apply(...), so that your backward() is used."
assert np.allclose(NETWORK_PROBE["network_forward"], [[0.610356], [0.561804]], atol=1e-5), \
    "The network's output is wrong. The forward pass should be layer1, ReLU, layer2, ReLU, output_layer, sigmoid."
print("3.1 local checks passed.")

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_custom_linear_function', answer=NN_PROBE)

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_custom_network', answer=NETWORK_PROBE)

## **3.2. Fit the Neural Network on the Diabetes Dataset [5 pts, Autograded]**

Train the network on the preprocessed training data and predict on the test set.

Everything except the training loop is provided. Write the loop: forward pass, loss, zero the
gradients, backward, step. Then put the model in evaluation mode, predict on `X_test_tensor`
under `torch.no_grad()`, and store the probabilities in `predictions_np` as a **1-D numpy array
of shape (N,)** — not `(N, 1)`. `numpy.squeeze` does this. The provided last line thresholds
it at 0.5.

> **Shape matters here.** The autograder scores `y_test_pred_NN` and rejects anything that is
> not 1-D. An `(N, 1)` array is the single most common reason for losing these points.

**How this is marked.** Every setting except the training loop is given to you, including the
random seed, so a correct loop ends up with the same network as ours. The autograder compares
your predictions with that network's, row by row, using
[Cohen's kappa](https://en.wikipedia.org/wiki/Cohen%27s_kappa): 1 means the two agree on every
row, and 0 means they agree no more often than chance. A correct training loop comes out at
about 1. Guessing at random comes out near 0, and so does predicting 0 for everyone, even
though that is 92% accurate on this data.

Marks: **5** — 1 for a 1-D integer prediction array of the right length, and 4 for
agreeing with the reference network: a kappa of at least 0.40 earns all 4, and 0.30, 0.20 and
0.10 earn 3, 2 and 1. The grading cell also tells you your accuracy and F1 on the test set,
which you'll need for Part 6.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

train_df_nn, test_df_nn = preprocess_datasets(*load_diabetes_datasets())

scaler = MinMaxScaler()
X_train = scaler.fit_transform(train_df_nn.iloc[:, :-1].values)
y_train = train_df_nn.iloc[:, -1].values
X_test = scaler.transform(test_df_nn.values)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

input_size = X_train_tensor.shape[1]
hidden_size1 = 64
hidden_size2 = 32
output_size = 1

torch.manual_seed(42)  # don't change this line!! -- re-seeded here so that re-running
                       # this cell gives the same network every time

# Instantiate the neural network
model = CustomNeuralNetwork(input_size, hidden_size1, hidden_size2, output_size)

# Define the loss function and optimizer.
#
# AdamW applies weight decay directly to the weights (L10). Bias terms are
# conventionally left unregularized (L07), so the parameters are split into two
# groups here and only the weights get weight_decay.
criterion = nn.BCELoss()
decay = [p for n, p in model.named_parameters() if not n.endswith("bias")]
no_decay = [p for n, p in model.named_parameters() if n.endswith("bias")]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.001},
     {"params": no_decay, "weight_decay": 0.0}], lr=0.01)

num_epochs = 1000

# Store the predicted probabilities in this variable, as a 1-D array of shape (N,)
predictions_np = None

for epoch in range(num_epochs):
    # STUDENT TODO START:
    # Forward pass, loss, optimizer.zero_grad(), loss.backward(), optimizer.step().
    raise NotImplementedError("The training loop is not implemented yet.")
    # STUDENT TODO END

# STUDENT TODO START:
# Put the model in eval mode, predict on X_test_tensor under torch.no_grad(),
# and store the probabilities in predictions_np as a 1-D numpy array.
raise NotImplementedError("Prediction is not implemented yet.")
# STUDENT TODO END

y_test_pred_NN = (predictions_np >= 0.5).astype(int)
assert y_test_pred_NN.ndim == 1,     f"y_test_pred_NN must be 1-D, got shape {y_test_pred_NN.shape}. Use numpy.squeeze()."
assert y_test_pred_NN.shape[0] == 2035,     f"Expected 2035 predictions, got {y_test_pred_NN.shape[0]}."
print("3.2 shape checks passed. Predicted positives:", int(np.sum(y_test_pred_NN)))

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_fit_and_predict_NN', answer=y_test_pred_NN)

# **4. Measures of Impurity and their Reduction [10 pts]**

> **This part and Part 5 are lectured during the homework window.** Impurity measures,
> information gain and pruning are covered in Decision Trees Part 2 on **Tue 13 October**,
> after this assignment is released. Everything you need is defined below, but if you would
> rather work from the lecture first, leave Parts 4 and 5 until then and do Parts 1–3 now.

To grow a classification tree, instead of a binary error (0/1), measures of impurity are used
to judge how good a node is. Lecture introduces **entropy** as one such measure; here you will
implement it and compare it with the **Gini index**.

## **4.1. Measures of Impurity [6 pts]**

For this problem, consider a binary classification problem with two classes: the positive
class $1$ and the negative class $0$. Both functions below take $p$, the probability that a
sample in the node belongs to class 1.

### **4.1.1. Entropy [2 pts, Autograded]**

Complete the `entropy` function. As in lecture, use $\log_2$:

$$H(p) = -p \log_2 p - (1-p)\log_2(1-p)$$

A node that is entirely one class has zero entropy. Note that $p\log_2 p \to 0$ as $p \to 0$,
so **`entropy(0)` and `entropy(1)` must both return `0.0`**, not `nan` — Part 4.2 calls this
function on pure child nodes.

In [ ]:
def entropy(prob_class1):
    """Entropy of a node, given the probability of class 1 in that node.

    Args:
        prob_class1: float in [0, 1]

    Returns:
        e: the entropy of the node, in bits. Returns 0.0 at p = 0 and p = 1.
    """
    # STUDENT TODO START:
    raise NotImplementedError("entropy is not implemented yet.")
    # STUDENT TODO END


# --- Local test. The hidden test checks the same things at a few more values of p.
assert np.abs(entropy(0.5) - 1.0) < 1e-9, "entropy(0.5) should be exactly 1 bit."
assert np.abs(entropy(0.25) - 0.811278124) < 1e-6, "Check the formula -- log base 2?"
assert entropy(0.0) == 0.0 and entropy(1.0) == 0.0,     "A pure node has zero entropy. Handle p = 0 and p = 1 without producing nan."
print("4.1.1 local test passed.")

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_entropy', answer=grader_serialize(entropy))

### **4.1.2. Gini Index [2 pts, Autograded]**

The Gini index is another measure of impurity. For a $C$-class problem, lecture writes it as

$$\text{Gini} = \sum_{c=1}^{C} p_c\,(1 - p_c)$$

Complete the function below for the binary case ($C = 2$).

In [ ]:
def gini_index(prob_class1):
    """Gini index of a node, given the probability of class 1 in that node.

    Args:
        prob_class1: float in [0, 1]

    Returns:
        gi: the Gini index of the node
    """
    # STUDENT TODO START:
    raise NotImplementedError("gini_index is not implemented yet.")
    # STUDENT TODO END


# --- Local test. The hidden test checks the same things at a few more values of p.
assert np.abs(gini_index(0.5) - 0.5) < 1e-9, "gini_index(0.5) should be 0.5."
assert np.abs(gini_index(0.25) - 0.375) < 1e-9, "Check the formula."
assert gini_index(0.0) == 0.0 and gini_index(1.0) == 0.0, "A pure node has zero impurity."
print("4.1.2 local test passed.")

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_gini_index', answer=grader_serialize(gini_index))

### **4.1.3. Impurity Measures Plot [2 pts, Manually Graded]**

Complete `impurity_measures_plot` to plot **entropy** and **Gini index** against the class-1
probability, both on the **same axes**. Give the plot a title, a legend and axis labels.

Marks: **1** for the plot, **1** for your written answer below it.

This section is graded by hand from the plot and the comment in this notebook.

In [ ]:
def impurity_measures_plot():
    """Plot entropy and Gini index against the class 1 probability, on one figure."""
    prob_class1_arr = np.arange(1, 1000) / 1000
    e_arr = np.array([entropy(p) for p in prob_class1_arr])
    gi_arr = np.array([gini_index(p) for p in prob_class1_arr])

    # STUDENT TODO START:
    raise NotImplementedError("impurity_measures_plot is not implemented yet.")
    # STUDENT TODO END


impurity_measures_plot()

**Your comment (4.1.3):** What do the two curves have in common that makes each of them a
sensible measure of impurity? Where is each one largest, where is it smallest, and what does
that mean for a node in a decision tree? Two or three sentences.

*replace this italic line with your answer*

## **4.2. Reduction in Impurity [4 pts]**

**Information gain** is the drop in entropy from a parent node to its children, with each
child weighted by the fraction of the parent's samples it receives:

$$IG = H(\text{parent}) - \sum_{v} \frac{n_v}{n} H(v)$$

**Gini reduction** is the same quantity with the Gini index in place of entropy.

Each parent below is split into exactly **two** children. You are given the parent's size and
class-1 count and the **first** child's size and class-1 count; the second child holds
whatever is left over. You may assume both children are non-empty.

### **4.2.1. Information Gain [2 pts, Autograded]**

Complete `information_gain`. Call your `entropy` function rather than re-deriving the formula.

In [ ]:
def information_gain(num_samples_parent, num_class1_parent, num_samples_child1, num_class1_child1):
    """Information gain of a binary split, in bits.

    The second child holds the remaining samples:
        num_samples_child2 = num_samples_parent - num_samples_child1
        num_class1_child2  = num_class1_parent  - num_class1_child1

    Args:
        num_samples_parent: number of samples in the parent node
        num_class1_parent: number of class 1 samples in the parent node
        num_samples_child1: number of samples in the first child node
        num_class1_child1: number of class 1 samples in the first child node

    Returns:
        ig: the information gain
    """
    # STUDENT TODO START:
    # 1. Compute the entropy of the parent and of each child, using entropy().
    # 2. Subtract the size-weighted average of the children's entropies from the parent's.
    raise NotImplementedError("information_gain is not implemented yet.")
    # STUDENT TODO END


# --- Local test. The hidden test uses the same formula on three other splits, one with a pure child.
assert np.abs(information_gain(100, 60, 30, 5) - 0.25123) < 0.01,     "Check the formula, and that each child is weighted by its share of the parent's samples."
assert np.abs(information_gain(100, 50, 40, 10) - 0.12451) < 0.01,     "Check the formula, and that each child is weighted by its share of the parent's samples."
print("4.2.1 local test passed.")

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_information_gain',
             answer=(grader_serialize(entropy), grader_serialize(information_gain)))

### **4.2.2. Gini Reduction [2 pts, Autograded]**

Complete `gini_reduction`, the same calculation with `gini_index` in place of `entropy`.

In [ ]:
def gini_reduction(num_samples_parent, num_class1_parent, num_samples_child1, num_class1_child1):
    """Gini reduction of a binary split.

    The second child holds the remaining samples, as in 4.2.1.

    Args:
        num_samples_parent: number of samples in the parent node
        num_class1_parent: number of class 1 samples in the parent node
        num_samples_child1: number of samples in the first child node
        num_class1_child1: number of class 1 samples in the first child node

    Returns:
        gr: the reduction in Gini index
    """
    # STUDENT TODO START:
    raise NotImplementedError("gini_reduction is not implemented yet.")
    # STUDENT TODO END


# --- Local test. The hidden test uses the same formula on three other splits, one with a pure child.
assert np.abs(gini_reduction(100, 60, 30, 5) - 0.16095) < 0.01,     "Check the formula, and that each child is weighted by its share of the parent's samples."
assert np.abs(gini_reduction(100, 50, 40, 10) - 0.08333) < 0.01,     "Check the formula, and that each child is weighted by its share of the parent's samples."
print("4.2.2 local test passed.")

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_gini_reduction',
             answer=(grader_serialize(gini_index), grader_serialize(gini_reduction)))

# **5. Decision Trees [12 pts — 5 for CIS 4190]**

Now you will train a decision tree classifier on the diabetes data from Part 2 and predict
who has diabetes.

> Like Part 4, this part is lectured on **Tue 13 October**, during the homework window.

Sections 5.2 and 5.3 are marked **[5190 Extra]**: required for CIS 5190, optional for CIS 4190.

## **5.1. Decision Tree Training with Pruning [5 pts, Autograded]**

`DecisionTreeClassifier` in scikit-learn prunes using **minimal cost-complexity pruning**. You
do not need the details — see the [documentation](https://scikit-learn.org/stable/modules/tree.html#minimal-cost-complexity-pruning)
if you are curious — because the amount of pruning depends entirely on `ccp_alpha`. You will
tune `ccp_alpha` by [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).

Implement `best_ccp_alpha_f1` to tune `ccp_alpha` automatically. sklearn has a
[built-in method](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.cost_complexity_pruning_path)
that generates the list of effective alphas to try.

This dataset is **imbalanced**: most respondents are not diabetic, so a model that predicts
"nobody has diabetes" scores about 92% accuracy while being useless. That is why we tune on
[F1](https://en.wikipedia.org/wiki/F-score) rather than accuracy. Use `scoring='f1'` with
5-fold cross-validation.

**Use `random_state=42` in every `DecisionTreeClassifier` you create**: the one that computes
the pruning path, the ones inside cross-validation, and the one you refit at the end. Without
it, the trees depend on whatever random state earlier cells left behind, so your answer can
change from one run to the next.

**`best_ccp_alpha_f1` returns two things**, and the second one matters for 5.2:
`cv_f1_scores` is the array of the **five per-fold F1 scores of the best alpha** — five
numbers, one per fold. It is *not* the list of mean scores across all the alphas you tried.

**How this is marked.** A decision tree with `random_state=42` is fully deterministic, so a
correct solution reproduces our tree's predictions exactly. As in 3.2, the autograder compares
your predictions with ours using Cohen's kappa (1 means they match on every row, 0 means no
better than chance).

Marks: **5** — 1 for a prediction array of the right shape and type, and 4 for agreeing
with the reference tree: a kappa of at least 0.90 earns all 4, and 0.70, 0.40 and 0.10 earn 3,
2 and 1. The grading cell also tells you your accuracy and F1 on the test set, which you'll
need for Part 6.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

train_df_dt, test_df_dt = preprocess_datasets(*load_diabetes_datasets())

def best_ccp_alpha_f1(train_df):
    """Tune ccp_alpha by 5-fold cross-validated F1.

    Args:
        train_df: the preprocessed training set, target in the last column

    Returns:
        best_ccp_alpha: the alpha with the highest mean cross-validated F1
        cv_f1_scores: np.ndarray of shape (5,) -- the five per-fold F1 scores
            AT best_ccp_alpha. Section 5.2 builds a confidence interval from
            these five numbers, so return five, not one per alpha tried.
    """
    # STUDENT TODO START:
    # Use random_state=42 in every DecisionTreeClassifier.
    raise NotImplementedError("best_ccp_alpha_f1 is not implemented yet.")
    # STUDENT TODO END


def refit_and_predict(train_df, test_df, best_ccp_alpha):
    """Refit on the whole training set at best_ccp_alpha and predict on the test set.

    Args:
        train_df: the preprocessed training set, target in the last column
        test_df: the preprocessed test features
        best_ccp_alpha: the tuned alpha

    Returns:
        y_test_pred: np.ndarray of shape (2035,), integer labels 0 or 1
    """
    # STUDENT TODO START:
    # Use random_state=42 here too.
    raise NotImplementedError("refit_and_predict is not implemented yet.")
    # STUDENT TODO END


best_ccp_alpha, cv_f1_scores = best_ccp_alpha_f1(train_df_dt)
y_test_pred_DT = refit_and_predict(train_df_dt, test_df_dt, best_ccp_alpha)

# --- Local test. These are the shape checks the hidden test makes first, plus some sanity
# checks. The hidden test then compares your predictions with the reference tree's.
assert np.size(cv_f1_scores) == 5,     (f"cv_f1_scores should hold the 5 per-fold scores at the best alpha, got "
     f"{np.size(cv_f1_scores)}. Re-read the docstring -- 5.2 depends on this.")
assert np.shape(y_test_pred_DT) == (2035,),     f"y_test_pred_DT should be 1-D of length 2035, got {np.shape(y_test_pred_DT)}."
assert set(np.unique(y_test_pred_DT)) == {0, 1}, \
    ("Your tree predicts only one class. If it's all 0s, check that ccp_alpha is tuned on F1, "
     "not accuracy: on data this imbalanced, accuracy rewards pruning the tree to a single leaf.")
assert np.mean(cv_f1_scores) < 0.6, \
    (f"cv_f1_scores average {np.mean(cv_f1_scores):.3f}, too high for F1 on this data. "
     "Make sure they're F1 scores (scoring='f1'), not accuracy.")
assert np.mean(cv_f1_scores) > 0.2, \
    (f"cv_f1_scores average {np.mean(cv_f1_scores):.3f}. The best alpha on the path should do "
     "better than that. Check that you pick the alpha with the highest mean F1.")
print("5.1 local test passed. best_ccp_alpha =", best_ccp_alpha,
      " mean cv F1 =", round(float(np.mean(cv_f1_scores)), 4))

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_refit_and_predict', answer=y_test_pred_DT)

---

> **CIS 4190 — sections 5.2 and 5.3 are [5190 Extra].** You're welcome to attempt them for
> credit, but your total is capped at 43 either way.
>
> If you skip 5.2, leave `calculate_confidence_interval` as it is and don't delete the cell.
> Its local test notices the function isn't written yet and prints a note instead of stopping,
> so **Run all** carries on to Part 6 and the two reports below: 6 points you still need.
> The 5.2 grading cell will report 0, which doesn't cost you anything.


## **5.2. Computing Confidence Intervals [5190 Extra — 3 pts, Autograded]**

You have just averaged the F1 score over five held-out folds. How confident can you be that
this average is near the classifier's true F1? Re-running with a different seed gives a
different number — so which one is right?

We will answer with a confidence interval based on Student's $t$-distribution, which says with
**99% confidence** that the true mean lies between a lower and an upper bound. From the sample
mean $\bar{x}$, the sample standard deviation $S$ and the number of observations $n$:

$$\bar{x} \pm t \cdot \frac{S}{\sqrt{n}}$$

$t$ is the critical value. Look it up in this
[t-table](https://www.stat.colostate.edu/inmem/gumina/st201/pdf/Utts-Heckard_t-Table.pdf) and
**round it to the hundredths place**. For example, for a 99% interval with $n = 10$, the 99%
column at $n - 1 = 9$ degrees of freedom gives $t = 3.25$.

**You may assume `len(cv_f1_scores) == 5`**, so $n = 5$, $n - 1 = 4$ degrees of freedom, and the
critical value is a single constant you read off the table — you do not need to write a general
lookup. **Do not use `scipy.stats`**: the point is to read the value off the table. A solution
that uses scipy gets no credit, and the local test below checks for it. (Mentioning scipy in a
comment is fine.)

Use the unbiased estimator of the variance, i.e. set `np.std`'s `ddof` argument.

Marks: **3** — 1 for returning a two-element array, 2 for the bounds being right.

In [ ]:
def calculate_confidence_interval(cv_f1_scores):
    """A 99% confidence interval for the mean of cv_f1_scores.

    Args:
        cv_f1_scores: np.ndarray of the five per-fold F1 scores

    Returns:
        interval: np.ndarray of shape (2,), [lower bound, upper bound]
    """
    # STUDENT TODO START:
    raise NotImplementedError("calculate_confidence_interval is not implemented yet.")
    # STUDENT TODO END


# --- Local test. Same tolerance as the hidden test, on a different sample. CIS 4190: if you
# skip 5.2, this cell prints a note instead of stopping, so Run all carries on.
try:
    _result = calculate_confidence_interval(np.array([15.6, 16.2, 22.5, 20.5, 16.4]))
except NotImplementedError:
    print("5.2 skipped: calculate_confidence_interval isn't written yet. Fine for CIS 4190.")
else:
    assert np.size(_result) == 2, f"Expected two bounds, got {np.size(_result)}."
    assert np.linalg.norm(np.array(_result) - np.array([11.918, 24.562])) <= 0.05, \
        ("Expected about [11.918, 24.562]. Check the t critical value you read off the table "
         "(99% column, 4 degrees of freedom), and that your standard deviation is the unbiased one.")
    assert not _code_uses(grader_serialize(calculate_confidence_interval), "scipy"), \
        "5.2 asks you to read t off the table, so a solution that uses scipy gets no credit."
    print("5.2 local test passed.")
    interval_DT = calculate_confidence_interval(cv_f1_scores)
    print("99% CI for the decision tree's cross-validated F1:", np.round(interval_DT, 4))

In [ ]:
# PennGrader Grading Cell
grader.grade(test_case_id='test_confidence_intervals',
             answer=grader_serialize(calculate_confidence_interval))

## **5.3. Performance Table [5190 Extra — 4 pts, Manually Graded]**

Repeat section 5.1 for **two more feature sets** of your own choosing, and present a table
comparing the three. Each set must include all 13 required features from section 2.2, plus
any of the other columns from the raw CSV that you want to try.

Do your working in the scratch cell below. Then fill in the table in the answer cell under it
(double-click that cell to edit it) and say which set you would pick.

Marks: **4** — 1 per additional feature set (alpha, mean F1 and interval filled in), 1 for
naming your choice and 1 for the reason.

In [ ]:
# Scratch space for 5.3. Nothing here is autograded -- copy your numbers into the
# table below. This cell is included so your working is visible to the grader.

# STUDENT TODO START:
# Build your two extra feature sets and run 5.1's pipeline on each.
# STUDENT TODO END

**Your answer (5.3):**

| S.No. | Features | Best CCP Alpha | Mean Cross-validation F1 Score | 99% Confidence Interval |
| --- | --- | --- | --- | --- |
| 1 | the 13 required features | | | |
| 2 | *describe set 2* | | | |
| 3 | *describe set 3* | | | |

**Your choice, and why:** *replace this italic line with your answer*

# **6. Which Model Would You Deploy? [2 pts, Manually Graded]**

You have now fitted two models to the same diabetes data: a **neural network** (Part 3) and a
**decision tree** (Part 5).

**You do not have the test labels** — the autograder holds them, which is what keeps the test
set honest. So base your comparison on evidence you do have:

* the cross-validated F1 from section 5.1 (and, if you did 5.2, its confidence interval)
* a validation split you hold out from the training data yourself, and score both models on
* the two models' accuracy and F1 as the grading cells reported them back to you

Which of the two would you deploy to screen patients for diabetes, and why? Say something
about the class imbalance, and about what a false negative costs compared with a false
positive in this setting.

Marks: **2** — 1 for a comparison backed by numbers you actually computed, 1 for the
reasoning about imbalance and error costs.

In [ ]:
# Scratch space for Part 6. Nothing here is autograded -- your answer goes in the
# markdown cell below, but leave your working here so the grader can see it.

# STUDENT TODO START:
# Hold out a validation split from the training data and score both models on it.
# STUDENT TODO END

**Your answer (6):**

*replace this italic line with your answer*

# **Before You Submit: Two Short Reports [4 pts, Autograded]**

These two cells are worth **2 points each**, and the points are for *answering*,
not for any particular answer. There is no answer that scores better than
another: reporting that you used no AI earns full marks, and so does reporting
that you used a lot of it.

We use the responses to calibrate how long assignments actually take and to
understand how students are using AI tools.

### **1. Use of AI tools [2 pts]**

Report any use of AI tools (ChatGPT, Claude, Copilot, Gemini, Cursor, etc.) while
working on this assignment: which ones, how you used them, and whether they helped
or got in the way of understanding the material.

**Being honest here will not cost you points.** Writing `"none"` earns full marks,
and so does describing heavy use. We are trying to learn what actually helps people
learn, which only works if the reports are accurate.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks --
# including "none". A few sentences is plenty.
AI_USE = ""


# PennGrader grading command
grader.grade('test_ai_use_report', AI_USE)

### **2. Time spent [2 pts]**

Roughly how long did this assignment take you, and how was that time split across
the parts? Rough numbers are genuinely useful — please do not agonize over precision.

You will get a far better estimate if you jot this down **as you go** rather than
reconstructing it at the end.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks.
# For example: "about 7 hours -- 1 on KNN, 3 on the neural network, 2 on the trees".
TIME_SPENT_HOURS = ""


# PennGrader grading command
grader.grade('test_time_report', TIME_SPENT_HOURS)

# **Submit to Gradescope**

Congratulations on completing the HW2 coding assignment!

**Before you download — check your figures and comments are there.** Sections 1.3, 4.1.3, 5.3
and 6 are marked by hand from the plots and the written answers in this notebook. Scroll back
and make sure every plot is showing and every comment cell has your own words in it. A
notebook downloaded with its outputs cleared arrives with nothing to mark in those sections,
and those points are more than a fifth of this assignment.

The final step is to download this iPython notebook and submit it to [Gradescope](https://www.gradescope.com).

*To download the notebook from Google Colab, click*
`File` → `Download` → `Download .ipynb`